
# Nonlinearities in Regression

## The World isn't Straight

So far, we have assumed that the relationship between variables is a straight line ($y = mx + c$). In the real world, relationships are rarely this simple.

-   Bacteria growth is **Exponential**.
-   Diminishing returns are **Logarithmic**.
-   Gravity and braking distances are **Quadratic** (Polynomial).

If we try to fit a straight line to a curved dataset, we face **Underfitting** (High Bias). The model is simply too rigid to capture the reality.

## The "Linear" Paradox

Surprisingly, to fit curves, we often still use **Linear Regression**. How?

We transform the **input data**, not the model. Consider a quadratic relationship: $y = \beta_0 + \beta_1 x + \beta_2 x^2$. If we create a new feature $z = x^2$, the equation becomes: $y = \beta_0 + \beta_1 x + \beta_2 z$.

This is now a **Linear Equation** with respect to the coefficients ($\beta$), even though it describes a curve with respect to $x$. This technique is called **Polynomial Regression**.

## Practical Demonstration: The Parabola

We will generate synthetic data representing a Parabolic/Quadratic trajectory and try to model it.

### Generate Non-Linear Data

$$y = 3x^2 + 8 + \text{noise}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Generate 100 samples from -3 to 3
n_samples = 100
X = np.linspace(-3, 3, n_samples).reshape(-1, 1)

# Quadratic equation with noise
y = 3 * X**2 + 8 + np.random.randn(n_samples, 1)

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='teal', alpha=0.6, label='Data')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Quadratic Data')
plt.legend()
plt.show()

### The Failure of Linear Regression

Let's see what happens if we stubbornly use a standard line.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Linear Model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Predict
y_pred_lin = linear_model.predict(X_test)

print(f"Linear R² Score: {r2_score(y_test, y_pred_lin):.2f}")
# Note: A low score is expected

### Success with Polynomial Features

We use `PolynomialFeatures` to generate $x^2$.

**Best Practice**: We use a scikit-learn `Pipeline`. This ensures that when we feed new data to the model, it automatically calculates the powers ($x^2, x^3$) before predicting.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Create a Pipeline:
# 1. Generate Polynomial Features (Degree 2)
# 2. Fit Linear Regression
poly_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())

# Train
poly_model.fit(X_train, y_train)

# Predict
y_pred_poly = poly_model.predict(X_test)

print(f"Polynomial (Deg 2) R² Score: {r2_score(y_test, y_pred_poly):.2f}")

### Visualizing the Difference

**Visualization Tip**: When plotting curves, we cannot just plot the Test Set points connected by lines, because they are in random order (it will look like spaghetti). We generate a sorted "smooth grid" of X values to draw the perfect regression curve.

In [ ]:
plt.figure(figsize=(10, 6))

# 1. Plot actual data
plt.scatter(X_test, y_test, color='gray', label='Test Data')

# 2. Create smooth grid for plotting regression curves
X_grid = np.linspace(-3, 3, 100).reshape(-1, 1)

# 3. Plot Linear Model
plt.plot(X_grid, linear_model.predict(X_grid), color='red', linewidth=2, label='Linear Fit')

# 4. Plot Polynomial Model
plt.plot(X_grid, poly_model.predict(X_grid), color='blue', linewidth=2, label='Polynomial Fit (Deg 2)')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Linear vs Polynomial Regression')
plt.legend()
plt.show()

## The Danger: Overfitting with High Degrees

If Degree 2 is good, is Degree 20 better? NO. High-degree polynomials create wild oscillations ("wiggles") to hit every single noisy data point. This is classic **Overfitting**.

In [ ]:
# Fit a Degree 20 model
high_degree_model = make_pipeline(PolynomialFeatures(degree=20), LinearRegression())
high_degree_model.fit(X_train, y_train)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='gray', alpha=0.3, label='Data')

# Plot the wiggle
y_wiggle = high_degree_model.predict(X_grid)
plt.plot(X_grid, y_wiggle, color='green', label='Degree 20 (Overfitting)')

plt.xlabel('x')
plt.ylabel('y')
plt.ylim(0, 40) # Limit y-axis to see the curve clearly
plt.title('The Danger of High Degrees')
plt.legend()
plt.show()

## Exercises

### The Cubic Challenge

Data follows a cubic rule: $y = x^3 - 3x$.

1.  Generate the data.
2.  Fit a Linear Model.
3.  Fit a Polynomial Model (Degree 3).
4.  Compare the $R^2$ scores.

### Regularized Polynomials

If you **must** use a high degree polynomial (e.g., Degree 10), you should use Regularization (Ridge/Lasso) to stop the wiggling.

1.  Create a `make_pipeline` with `PolynomialFeatures(degree=10)` and `Ridge(alpha=100)`.
2.  Fit it to the cubic data above.
3.  Observe if it is smoother than a standard Linear Regression with Degree 10.

## Summary

1.  **Linearity in Parameters**: We can fit curves using Linear Regression by squaring/cubing the features.
2.  **PolynomialFeatures**: The sklearn tool that creates these powers for us.
3.  **Pipelines**: The clean way to package transformation + modeling.
4.  **Balance**: Low degree = Underfitting (Lines). High degree = Overfitting (Wiggles).